In [ ]:
#| default_exp quantize.adaround

In [ ]:
#| export
from __future__ import annotations
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.fusion import fuse_conv_bn_eval

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

`adaround_quantize` applies **AdaRound** (Nagel et al., 2020, *"Up or Down? Adaptive Rounding for
Post-Training Quantization"*) — the learned-rounding criterion that makes **INT4 weight
quantization** viable for redundant CNNs.

Naive round-to-nearest (RTN) rounds every weight to its closest grid point independently. At INT4
that greedy choice is *jointly* far from optimal: small per-weight rounding errors accumulate
through the layer and collapse accuracy (the "INT4 cliff"). AdaRound instead **learns**, per layer,
whether each weight should round *up* or *down* — minimising the layer's output reconstruction error
`‖W_q x − W x‖²` on a little calibration data. It recovers most of the INT4 cliff on ResNet-class
networks (≈ INT8, ~1 pt) while leaving depthwise-heavy nets (MobileNet) architecture-dependently
lossy — so INT4 is a *size* lever for redundant CNNs, not a universal one.

### How it maps to the dimensional `Quantizer`

This is the first concrete criterion built toward the dimensional `Quantizer`
(see `DIMENSIONAL_DESIGN.md`). In that grammar it is `criteria='adaround'`:

| dimension | value here |
|---|---|
| **granularity** | `'channel'` (per-output-channel) or `'tensor'` |
| **criteria** | `adaround` — learned rounding (this notebook) |
| **precision** | `w_bits` weights, optional `act_bits` activations |
| **schedule** | `None` — one-shot PTQ |

The returned model is an **accuracy-faithful fake-quantized** model (weights are quantized then
de-quantized in-place). Real INT4-kernel export (torchao / ONNX-QDQ) is the deployment follow-up.

In [ ]:
#| export
def _set_child(parent, name, new):
    "Replace child module `name` of `parent` (handles Sequential integer indices)."
    if name.isdigit(): parent[int(name)] = new
    else: setattr(parent, name, new)

def fold_bn(model: nn.Module   # model containing Conv2d→BatchNorm2d pairs
    ) -> nn.Module:            # same model with each Conv-BN pair fused in-place
    "Fuse each `Conv2d`→`BatchNorm2d` pair into the conv (eval mode) and replace the BN with Identity."
    for mod in model.modules():
        kids = list(mod.named_children())
        for i in range(len(kids) - 1):
            (n1, c1), (n2, c2) = kids[i], kids[i + 1]
            if isinstance(c1, nn.Conv2d) and isinstance(c2, nn.BatchNorm2d):
                _set_child(mod, n1, fuse_conv_bn_eval(c1, c2))
                _set_child(mod, n2, nn.Identity())
    return model

In [ ]:
#| export
def weight_scale(W: torch.Tensor,             # weight tensor [out_channels, ...]
                 w_bits: int = 4,             # bit-width (signed)
                 granularity: str = 'channel',  # 'channel' (per-output-channel) or 'tensor'
    ):                                        # (scale, qmin, qmax) — dequant scale and signed clip range
    "Signed weight quantization `scale` and clip range `[qmin, qmax]` on the ONNX/ORT grid."
    qmax =   2 ** (w_bits - 1) - 1                     # INT4 ->  7,  INT8 ->  127
    qmin = -(2 ** (w_bits - 1))                        # INT4 -> -8,  INT8 -> -128 (asymmetric signed grid)
    denom =  2 ** (w_bits - 1) - 0.5                   # INT4 -> 7.5  (matches onnxruntime per-channel int4)
    if granularity == 'channel':
        r = W.detach().abs().reshape(W.shape[0], -1).amax(1).clamp_min(1e-12)
        scale = (r / denom).reshape([-1] + [1] * (W.dim() - 1))
    else:
        scale = W.detach().abs().amax().clamp_min(1e-12) / denom
    return scale, qmin, qmax

def rtn_quant(W: torch.Tensor,                # weight tensor to fake-quantize
              w_bits: int = 4,                # bit-width
              granularity: str = 'channel',   # 'channel' or 'tensor'
    ) -> torch.Tensor:                        # dequantized (fake-quant) weights, same shape
    "Round-to-nearest (RTN) signed fake-quant — the AdaRound baseline (== AdaRound at init)."
    scale, qmin, qmax = weight_scale(W, w_bits, granularity)
    floorWs = torch.floor(W / scale)
    q = floorWs + ((W / scale - floorWs) >= 0.5).float()  # round half up (matches the AdaRound hard round)
    return torch.clamp(q, qmin, qmax) * scale

In [ ]:
#| export
# AdaRound rectified-sigmoid soft rounding (Nagel et al., 2020):
#   h(V) = clamp(sigmoid(V)*(zeta-gamma)+gamma, 0, 1)
#   W_q(V) = s * clamp(floor(W/s) + h(V), qmin, qmax)
_ZETA, _GAMMA = 1.1, -0.1

def _rect_sigmoid(V: torch.Tensor) -> torch.Tensor:
    "Rectified sigmoid squashing `V` into a soft rounding decision in [0, 1]."
    return torch.clamp(torch.sigmoid(V) * (_ZETA - _GAMMA) + _GAMMA, 0, 1)

def _init_V(frac: torch.Tensor) -> torch.Tensor:
    "Initialise `V` so that `h(V) == frac(W/s)` — makes AdaRound identical to RTN at init."
    s0 = ((frac - _GAMMA) / (_ZETA - _GAMMA)).clamp(1e-6, 1 - 1e-6)
    return torch.log(s0 / (1 - s0))

def _hard_quant_from_V(V, floorWs, scale, qmin, qmax):
    "Deployed weights: hard-round the learned soft rounding `h(V)` onto the signed grid."
    h_hard = (_rect_sigmoid(V) >= 0.5).float()
    return scale * torch.clamp(floorWs + h_hard, qmin, qmax)

def _layer_fwd(mod, x, w, b):
    "Forward one Conv2d/Linear layer with an explicit weight/bias (for reconstruction)."
    if isinstance(mod, nn.Conv2d):
        return F.conv2d(x, w, b, mod.stride, mod.padding, mod.dilation, mod.groups)
    return F.linear(x, w, b)

In [ ]:
#| export
def adaround_layer(layer: nn.Module,             # Conv2d/Linear whose rounding is optimized
                   x_cal: torch.Tensor,          # cached FP inputs to this layer [N, ...]
                   w_bits: int = 4,              # weight bit-width
                   *,
                   granularity: str = 'channel',  # 'channel' or 'tensor'
                   iters: int = 2000,            # optimization iterations
                   lr: float = 1e-2,             # Adam learning rate
                   lam: float = 0.01,            # rounding-regularizer weight
                   beta_start: float = 20.0,     # anneal start (encourages soft)
                   beta_end: float = 2.0,        # anneal end (encourages binary)
                   warmup: float = 0.2,          # fraction of iters before regularizer turns on
                   batch_size: int = 32,         # minibatch of cached inputs per step
                   device: str | torch.device | None = None,  # compute device (default: layer's)
    ):                                           # (Wq_hard, info) — deployed fake-quant weights + diagnostics
    "Learn per-weight soft rounding for one layer, minimising output reconstruction ‖W_q(V)x − Wx‖² + regularizer."
    device = torch.device(device) if device is not None else layer.weight.device
    W = layer.weight.detach().float().to(device)
    scale, qmin, qmax = weight_scale(W, w_bits, granularity)
    floorWs = torch.floor(W / scale)
    frac = (W / scale) - floorWs                          # in [0, 1)
    V = _init_V(frac).clone().requires_grad_(True)        # init so h(V) == frac  (== RTN)
    opt = torch.optim.Adam([V], lr=lr)
    bias = layer.bias.detach().float().to(device) if layer.bias is not None else None
    N = x_cal.shape[0]
    warm_iters = int(warmup * iters)
    for it in range(iters):
        idx = torch.randint(0, N, (min(batch_size, N),))
        xb = x_cal[idx].to(device, dtype=torch.float32)
        h = _rect_sigmoid(V)
        Wq = scale * torch.clamp(floorWs + h, qmin, qmax)
        out_q = _layer_fwd(layer, xb, Wq, bias)
        with torch.no_grad():
            out_fp = _layer_fwd(layer, xb, W, bias)
        rec = (out_q - out_fp).pow(2).flatten(1).sum(1).mean()  # per-sample SSE, averaged
        if it < warm_iters:
            beta, lam_t = beta_start, 0.0                       # warmup: reconstruction only
        else:
            rel = (it - warm_iters) / max(1, iters - warm_iters)
            beta = beta_end + (beta_start - beta_end) * max(0.0, 1 - rel)
            lam_t = lam
        round_loss = lam_t * (1 - (2 * h - 1).abs().pow(beta)).sum()  # push h -> {0, 1}
        loss = rec + round_loss
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        h = _rect_sigmoid(V)
        not_binary = ((h > 0.02) & (h < 0.98)).float().mean().item()  # unresolved fraction
        Wq_hard = _hard_quant_from_V(V, floorWs, scale, qmin, qmax)
    return Wq_hard, {'not_binary_frac': not_binary, 'w_bits': w_bits, 'granularity': granularity}

In [ ]:
#| export
def _iter_target_layers(model: nn.Module,                     # model to scan
                        layer_type = (nn.Conv2d, nn.Linear),  # module type(s) to target
    ):
    "Iterate over modules of `layer_type` (the AdaRound analogue of `Sparsifier._iter_layers`)."
    for m in model.modules():
        if isinstance(m, layer_type):
            yield m

def _normalize_calibration(calibration_dl,               # DataLoaders | DataLoader | list of batches
                           n_batches: int | None = None,  # cap on number of batches
    ) -> list:                                           # list of input tensors (xb)
    "Normalise the three accepted calibration inputs to a list of input tensors."
    dl = calibration_dl
    if hasattr(dl, 'valid') and hasattr(dl, 'train'):    # fastai DataLoaders -> use validation dl
        dl = dl.valid
    batches = []
    for b in dl:
        xb = b[0] if isinstance(b, (list, tuple)) else b  # (xb, yb) batch or a bare input tensor
        batches.append(xb)
        if n_batches is not None and len(batches) >= n_batches:
            break
    return batches

@torch.no_grad()
def _capture_inputs(model,                          # (pristine) FP model
                    layers,                         # target layers to hook
                    batches,                        # list of input tensors
                    device,                         # compute device
                    store_dtype = torch.float16,    # cache dtype (saves memory)
    ) -> dict:                                      # {layer: cached inputs [N, ...] on CPU}
    "Cache the FP input tensor to every target layer in a single forward pass."
    cache = {l: [] for l in layers}
    def make_hook(layer):
        def hook(mod, inp):
            cache[layer].append(inp[0].detach().to(store_dtype).cpu())
        return hook
    handles = [l.register_forward_pre_hook(make_hook(l)) for l in layers]
    model.eval()
    for xb in batches:
        model(xb.to(device))
    for h in handles: h.remove()
    return {l: torch.cat(v) for l, v in cache.items()}

In [ ]:
#| export
def _fake_quant_act(x, scale, qmax):
    "Per-tensor symmetric fake-quant of an activation tensor."
    return torch.clamp(torch.round(x / scale), -qmax, qmax) * scale

@torch.no_grad()
def _calibrate_activations(model,           # weight-quantized model
                           layers,          # target layers
                           batches,         # calibration input tensors
                           act_bits: int,   # activation bit-width
                           device,          # compute device
                           n_batches: int = 8,  # batches used for statistics
    ):
    "Calibrate MSE-optimal per-tensor activation scales and install persistent fake-quant pre-hooks."
    qmax = 2 ** (act_bits - 1) - 1
    samples = {l: [] for l in layers}
    def obs(mod, inp):
        v = inp[0].detach().flatten()
        if v.numel() > 20000:
            v = v[torch.randint(0, v.numel(), (20000,), device=v.device)]
        samples[mod].append(v.float().cpu())
    handles = [l.register_forward_pre_hook(obs) for l in layers]
    model.eval()
    for i, xb in enumerate(batches):
        if i >= n_batches: break
        model(xb.to(device))
    for h in handles: h.remove()
    alphas = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5, 0.4, 0.3]
    def make_hook(scale, qmax):
        def hook(mod, inp):
            return (_fake_quant_act(inp[0], scale, qmax),) + tuple(inp[1:])
        return hook
    for l in layers:
        a = torch.cat(samples[l]); mx = a.abs().max().item()
        best_mse, best_clip = float('inf'), mx
        for al in alphas:                                 # search clip minimising quant MSE
            clip = al * mx
            if clip <= 0: continue
            scale = clip / qmax
            q = torch.clamp(torch.round(a / scale), -qmax, qmax) * scale
            mse = ((q - a) ** 2).mean().item()
            if mse < best_mse: best_mse, best_clip = mse, clip
        l.register_forward_pre_hook(make_hook(max(best_clip, 1e-12) / qmax, qmax))

In [ ]:
#| export
def adaround_quantize(
    model: nn.Module,                              # model whose weights -> INT-`w_bits` with learned rounding
    calibration_dl,                                # fastai DataLoaders | torch DataLoader | list of (xb, yb)/xb batches -- REQUIRED
    w_bits: int = 4,                               # weight bit-width
    granularity: str = 'channel',                  # 'channel' (per-output-channel) or 'tensor'
    act_bits: int | None = None,                   # activation bit-width (None -> weight-only)
    iters: int = 2000,                             # AdaRound optimization iters per layer
    lr: float = 1e-2,                              # Adam learning rate
    fold_batchnorm: bool = True,                   # fuse Conv->BN before quantizing
    layer_type = (nn.Conv2d, nn.Linear),           # module type(s) to quantize
    *,
    lam: float = 0.01,                             # AdaRound rounding-regularizer weight
    batch_size: int = 32,                          # minibatch of cached inputs per AdaRound step
    n_calib_batches: int | None = None,            # cap on calibration batches used
    device: str | torch.device | None = None,      # compute device (default: model's)
    verbose: bool = False,                         # print per-layer progress
) -> nn.Module:                                    # AdaRound fake-quantized model
    """Post-training INT weight quantization with learned rounding (AdaRound, Nagel et al. 2020).

    Signed per-output-channel (or per-tensor) fake-quant whose rounding is *learned* per layer by
    minimising output reconstruction error, recovering most of the INT4 round-to-nearest cliff for
    redundant CNNs (ResNet-class ~ INT8; depthwise-heavy nets stay architecture-dependently lossy).

    Returns an accuracy-faithful **fake-quantized** model (weights quantized then de-quantized
    in-place). Real INT4-kernel export (torchao / ONNX-QDQ) is the deployment follow-up. In the
    dimensional `Quantizer` this is `criteria='adaround'`."""
    if calibration_dl is None:
        raise ValueError("adaround_quantize requires calibration data -- pass a DataLoaders, a "
                         "DataLoader, or a list of input batches as `calibration_dl` (got None).")
    if granularity not in ('channel', 'tensor'):
        raise ValueError(f"granularity must be 'channel' or 'tensor', got {granularity!r}.")
    if not 2 <= w_bits <= 8:
        raise ValueError(f"w_bits must be between 2 and 8, got {w_bits}.")
    if act_bits is not None and not 2 <= act_bits <= 8:
        raise ValueError(f"act_bits must be between 2 and 8 (or None), got {act_bits}.")

    qmodel = copy.deepcopy(model).eval()
    device = torch.device(device) if device is not None else next(qmodel.parameters()).device
    qmodel = qmodel.to(device)
    if fold_batchnorm: fold_bn(qmodel)

    batches = _normalize_calibration(calibration_dl, n_calib_batches)
    if len(batches) == 0:
        raise ValueError("calibration_dl yielded no batches.")

    layers = list(_iter_target_layers(qmodel, layer_type))
    caches = _capture_inputs(qmodel, layers, batches, device)  # FP inputs (parallel AdaRound)

    for i, layer in enumerate(layers):
        Wq, info = adaround_layer(layer, caches[layer], w_bits, granularity=granularity,
                                  iters=iters, lr=lr, lam=lam, batch_size=batch_size, device=device)
        layer.weight.data = Wq.to(layer.weight.dtype)
        if verbose:
            print(f"[{i+1}/{len(layers)}] {type(layer).__name__} "
                  f"not_binary={info['not_binary_frac']:.3f}")

    if act_bits is not None:
        _calibrate_activations(qmodel, layers, batches, act_bits, device)
    return qmodel

In [ ]:
show_doc(adaround_quantize)

In [ ]:
show_doc(rtn_quant)

In [ ]:
show_doc(adaround_layer)

---

## Usage

```python
from fasterai.quantize.adaround import adaround_quantize

# Weight-only INT4 with learned rounding, per-output-channel scales
qmodel = adaround_quantize(
    model,
    calibration_dl=dls.valid,   # a fastai DataLoaders, a torch DataLoader, or a list of batches
    w_bits=4,
    granularity='channel',
    iters=2000,
)

# W4A8 (also fake-quantize activations to INT8)
qmodel = adaround_quantize(model, dls.valid, w_bits=4, act_bits=8)
```

`calibration_dl` accepts three shapes, normalised internally:

- a **fastai `DataLoaders`** (its `.valid` loader is used),
- a **torch `DataLoader`** yielding `(xb, yb)` batches, or
- a plain **list of batches** — each a `(xb, yb)` tuple or a bare input tensor.

In [ ]:
#| hide
from fastcore.test import *
torch.manual_seed(0)

def _tiny_net():
    "Small 2-conv + 1-linear net used across the fast tests."
    return nn.Sequential(
        nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
        nn.Conv2d(8, 16, 3, padding=1), nn.AdaptiveAvgPool2d(1),
        nn.Flatten(), nn.Linear(16, 4),
    ).eval()

_calib = [torch.randn(4, 3, 16, 16) for _ in range(3)]  # list-of-batches calibration

# 1) SANITY GATE (correctness): at init (iters=0) AdaRound weights == RTN weights,
#    because V is initialised so h(V) == frac(W/s). If this fails, the port is wrong.
_net = _tiny_net()
_q0 = adaround_quantize(copy.deepcopy(_net), _calib, w_bits=4, iters=0,
                        fold_batchnorm=False, device='cpu')
_orig = [m for m in _net.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
_q0l  = [m for m in _q0.modules()  if isinstance(m, (nn.Conv2d, nn.Linear))]
test_eq(len(_q0l), 3)  # 2 conv + 1 linear
for o, q in zip(_orig, _q0l):
    test_close(q.weight, rtn_quant(o.weight.detach(), 4, 'channel'), eps=1e-5)

In [ ]:
#| hide
# 2) RECONSTRUCTION IMPROVES: after a short AdaRound run the per-layer reconstruction
#    error ‖W_q x − W x‖² is <= the RTN reconstruction error.
torch.manual_seed(0)
_conv = nn.Conv2d(4, 8, 3, padding=1).eval()
_x = torch.randn(32, 4, 8, 8)
_W = _conv.weight.detach()
with torch.no_grad():
    _out_fp = F.conv2d(_x, _W, _conv.bias, _conv.stride, _conv.padding)
    _Wq_rtn = rtn_quant(_W, 4, 'channel')
    _rec_rtn = (F.conv2d(_x, _Wq_rtn, _conv.bias, _conv.stride, _conv.padding) - _out_fp).pow(2).sum().item()
_Wq_ada, _info = adaround_layer(_conv, _x, 4, iters=400, batch_size=16, device='cpu')
with torch.no_grad():
    _rec_ada = (F.conv2d(_x, _Wq_ada, _conv.bias, _conv.stride, _conv.padding) - _out_fp).pow(2).sum().item()
assert _rec_ada <= _rec_rtn + 1e-6, f"AdaRound rec {_rec_ada:.4g} should be <= RTN rec {_rec_rtn:.4g}"
# init V so h(V)=frac -> diagnostics dict is well-formed
test_eq(set(_info), {'not_binary_frac', 'w_bits', 'granularity'})

In [ ]:
#| hide
# 4) FAILURE MODES / USER-SIDE INPUTS
_net = _tiny_net()
# Missing calibration data -> clear ValueError mentioning calibration
with ExceptionExpected(ValueError, regex='calibration'):
    adaround_quantize(_net, None, w_bits=4)
# Unsupported granularity -> clear error
with ExceptionExpected(ValueError, regex='granularity'):
    adaround_quantize(_net, _calib, granularity='block')
# Unsupported w_bits -> clear error
with ExceptionExpected(ValueError, regex='w_bits'):
    adaround_quantize(_net, _calib, w_bits=1)
# Unsupported act_bits -> clear error
with ExceptionExpected(ValueError, regex='act_bits'):
    adaround_quantize(_net, _calib, w_bits=4, act_bits=1)

# A plain LIST OF BATCHES works end-to-end (cross-function consistency, not only a DataLoaders)
_qm = adaround_quantize(copy.deepcopy(_net), _calib, w_bits=4, iters=5,
                        fold_batchnorm=False, device='cpu')
_o = _qm(torch.randn(2, 3, 16, 16))
test_eq(_o.shape, (2, 4))
assert torch.isfinite(_o).all()
# Weight-only leaves the model callable; act_bits installs activation fake-quant hooks
_qm_a = adaround_quantize(copy.deepcopy(_net), _calib, w_bits=4, act_bits=8, iters=5,
                          fold_batchnorm=False, device='cpu')
assert torch.isfinite(_qm_a(torch.randn(2, 3, 16, 16))).all()
# 'tensor' granularity is a valid alternative to 'channel'
_qm_t = adaround_quantize(copy.deepcopy(_net), _calib, w_bits=4, granularity='tensor',
                          iters=5, fold_batchnorm=False, device='cpu')
assert torch.isfinite(_qm_t(torch.randn(2, 3, 16, 16))).all()

In [ ]:
#| hide
#| slow
# 3) BEHAVIORAL RECOVERY (real model, real data): INT4+AdaRound top-1 > INT4-RTN top-1.
#    Small Imagenette slice, ResNet-18 pretrained on ImageNet (1000-way argmax).
import os, torchvision
from torchvision import transforms

_DATA = '/home/nathan/.fastai/data/imagenette2-320'
# Imagenette WNID (alphabetical = ImageFolder idx 0..9) -> ImageNet class index
_WNID2IN = {'n01440764': 0, 'n02102040': 217, 'n02979186': 482, 'n03000684': 491,
            'n03028079': 497, 'n03394916': 566, 'n03417042': 569, 'n03425413': 571,
            'n03445777': 574, 'n03888257': 701}
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'

_tfm = transforms.Compose([
    transforms.Resize(232, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
_ds = torchvision.datasets.ImageFolder(os.path.join(_DATA, 'val'), transform=_tfm)
_i2w = {v: k for k, v in _ds.class_to_idx.items()}
_remap = {i: _WNID2IN[_i2w[i]] for i in range(len(_i2w))}
_dl = torch.utils.data.DataLoader(_ds, batch_size=64, shuffle=True, num_workers=8)
_xs, _ys = [], []
for _xb, _yb in _dl:
    _xs.append(_xb); _ys.append(torch.tensor([_remap[int(t)] for t in _yb]))
    if sum(x.shape[0] for x in _xs) >= 384: break
_X = torch.cat(_xs)[:384]; _Y = torch.cat(_ys)[:384]
_calib_rn = [_X[i:i+32] for i in range(0, 128, 32)]      # 128 calibration images
_Xe, _Ye = _X[128:], _Y[128:]                            # 256 eval images

_rn = torchvision.models.resnet18(weights='IMAGENET1K_V1').eval().to(_dev)
fold_bn(_rn)

@torch.no_grad()
def _top1(m):
    m.eval(); ok = 0
    for i in range(0, _Xe.shape[0], 64):
        ok += (m(_Xe[i:i+64].to(_dev)).argmax(1).cpu() == _Ye[i:i+64]).sum().item()
    return 100.0 * ok / _Xe.shape[0]

_acc_fp = _top1(_rn)

_rtn = copy.deepcopy(_rn)                                 # INT4 RTN baseline (weight-only)
for _l in _iter_target_layers(_rtn):
    _l.weight.data = rtn_quant(_l.weight.detach().float(), 4, 'channel').to(_l.weight.dtype)
_acc_rtn = _top1(_rtn)

_ada = adaround_quantize(_rn, _calib_rn, w_bits=4, iters=1000, device=_dev)  # INT4 + AdaRound
_acc_ada = _top1(_ada)

print(f"ResNet-18 Imagenette top-1  |  FP={_acc_fp:.1f}  INT4-RTN={_acc_rtn:.1f}  INT4-AdaRound={_acc_ada:.1f}")
assert _acc_ada > _acc_rtn, f"AdaRound {_acc_ada:.1f} should recover over RTN {_acc_rtn:.1f}"

---

## See Also

- [Dimensional Quantizer design](DIMENSIONAL_DESIGN.md) — the north-star API. AdaRound is
  `criteria='adaround'` in that grammar (composable with `granularity`, `precision`, `context`).
- [Quantizer](quantizer.html) — backend-oriented INT8 PTQ/QAT (torch.ao / torchao).
- [QuantizeCallback](quantize_callback.html) — apply quantization during fastai training.
- [BN Folding](../misc/bn_folding.html) — the Conv→BN fusion AdaRound runs first.